In [11]:
from langgraph.checkpoint.memory import InMemorySaver, MemorySaver
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel, branch
from pydantic import BaseModel, Field

from  langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import Literal, TypedDict, List, Annotated
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from IPython.display import Image,display, Markdown
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper



# This function will load all the variables from the .env file and will
# make them available in the os.environ dictionary (env variables)
load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()

class llm_schema(BaseModel):
    funny_flag: Literal["funny","not funny"] = Field(description="Whether the joke is funny or not")
    feedback: str = Field(description="Feedback on the joke")

llm_with_schema = llm_openai.with_structured_output(llm_schema)


class graph_schema(TypedDict):
    messages: Annotated[List, add_messages]

def welcome(schema:graph_schema) -> graph_schema:

    curr_message = schema['messages']

    response = llm_openai.invoke(curr_message).content

    schema['messages'] = f"Your message was {curr_message}. Here's my response:  {response}"

    return schema


graph=StateGraph(graph_schema)

# Adding Nodes
graph.add_node("welcome", welcome)

# Adding Edges
graph.add_edge(START, "welcome")
graph.add_edge("welcome", END)

checkpoint = MemorySaver()




f_graph=graph.compile(checkpointer=checkpoint)

# Display Mermaid text instead of trying to render as PNG
print(f_graph.get_graph().draw_mermaid())

response = f_graph.invoke(
    {"messages":"What is my name?"},
    {'configurable':{'thread_id': 'ujjwal_manna'}}
)

for message in response['messages']:
    message.pretty_print()

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	welcome(welcome)
	__end__([<p>__end__</p>]):::last
	__start__ --> welcome;
	welcome --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

================================ Human Message =================================

What is my name?
================================ Human Message =================================

Your message was [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='5c28a5fd-2517-4e04-8069-9b26cfa16a3f')]. Here's my response:  I don’t have any information about your name. If you’d like to share it, feel free to let me know!


In [12]:
response = f_graph.invoke(
    {"messages":"My name ujjwal"},
    {'configurable':{'thread_id': 'ujjwal_manna'}}
)
for message in response['messages']:
    message.pretty_print()


================================ Human Message =================================

What is my name?
================================ Human Message =================================

Your message was [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='5c28a5fd-2517-4e04-8069-9b26cfa16a3f')]. Here's my response:  I don’t have any information about your name. If you’d like to share it, feel free to let me know!
================================ Human Message =================================

My name ujjwal
================================ Human Message =================================

Your message was [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='5c28a5fd-2517-4e04-8069-9b26cfa16a3f'), HumanMessage(content="Your message was [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='5c28a5fd-2517-4e04-8069-9b26cfa16a3f')]. Here's my response:  I don’t have any information about